# TP3 — Gestión integrada de stocks (padrón terminado en 5)

Notebook de ejecución para Colab o Jupyter. Usa los módulos de `src/` (mismo código que `main.py`).

**En Colab:** subir y descomprimir el ZIP del TP y ubicarse en la carpeta `tp3` (celda siguiente).

In [ ]:
# En Colab, descomentar y ajustar:
# !unzip -q TP3_Stocks_padron5.zip
# %cd tp3
%pip install -q -r requirements.txt

In [ ]:
import sys, warnings, numpy as np, pandas as pd
warnings.filterwarnings('ignore')
sys.path.insert(0, 'src')
from parametros import Caso
from pronostico import cargar_datos, pronosticar, generar_escenarios
from modelo import generar_arbol, construir_y_resolver, resolver_valor_esperado
from simulacion import simular, nivel_servicio, resumen_resultado
caso = Caso()
caso.resumen()

## Parte A — Pronóstico
Validación de origen móvil (1 y 2 meses adelante) y selección por MAPE.

In [ ]:
df = cargar_datos()
fc = pronosticar(df, cv_min_int=caso.cv_min_demanda_interna)
for s in ('di','de','tc'):
    print(s, fc[s]['metodo'], np.round(fc[s]['pron'],1), 'desvío relativo', np.round(fc[s]['sd_rel'],4))
    display(fc[s]['tabla'].round(3))

## Parte B — Programa lineal estocástico (PuLP)
Árbol de 100 nodos de noviembre × 50 de diciembre.

In [ ]:
arbol = generar_arbol(fc, 100, 50, np.random.default_rng(2026))
opt = construir_y_resolver(caso, arbol)
{k: v for k, v in opt.items() if k != 'modelo'}

In [ ]:
plan = {'Jn': float(round(opt['Jn'])), 'Pd': round(opt['C'])/1.5, 'Jd': float(round(opt['Jd'])),
        'F': float(np.floor(min(opt['F'], caso.maiz_max_anticipo)*100)/100)}
print('Formulario:', plan['Jn'], round(plan['Pd']*1.5,2), plan['Jd'], plan['F'])

## Evaluación fuera de muestra y Parte C.1

In [ ]:
esc = generar_escenarios(fc, 20000, np.random.default_rng(12345))
sim = simular(caso, plan, esc)
print(resumen_resultado(sim))
nivel_servicio(sim).round(3)

## Análisis completo (C.2, C.3, robustez, sensibilidad, gráficos e informe)
Ejecuta el script principal y luego genera el informe.

In [ ]:
!python main.py
!python informe/generar_informe.py